# nuScenes full ego-motion profile: no model training


In [ ]:
"""Deterministic census of six-keyframe ego motion; Python standard library only."""
import csv
import gc
import hashlib
import json
import math
import os
import platform
import statistics
import subprocess
import sys
import tarfile
import time
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

CLAIM_ID = "claim-nuscenes-three-second-low-lateral-displacement-majority"
OFFICIAL_META_URL = "https://motional-nuscenes.s3.amazonaws.com/public/v1.0/v1.0-trainval_meta.tgz"
TABLES = ("scene.json", "sample.json", "sample_data.json", "ego_pose.json")
PROTOCOL = {
    "horizon_keyframes": 6,
    "nominal_horizon_seconds": 3,
    "primary_abs_endpoint_lateral_lt_m": 2.0,
    "majority_ratio_strictly_gt": 0.5,
    "stopped_path_length_le_m": 0.3,
    "low_speed_path_length_le_m": 1.5,
    "crosscheck_min_each_step_gt_m": 0.3,
    "crosscheck_accumulated_difference_gt_deg": 15.0,
    "lateral_sensitivity_m": [0.5, 1.0, 2.0, 3.0],
    "heading_sensitivity_deg": [3, 5, 10, 15, 20, 30],
    "design_origin": "post-exploration definition frozen before this follow-up census; not the original preregistration",
}


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def json_write(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def wrap(angle):
    return (angle + math.pi) % (2 * math.pi) - math.pi


def quat_yaw(q):
    assert len(q) == 4 and all(math.isfinite(v) for v in q)
    assert abs(sum(v * v for v in q) - 1) < 1e-3, "Non-unit ego quaternion"
    w, x, y, z = q
    return math.atan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))


def motion_metrics(poses):
    """poses: seven (x, y, yaw, timestamp_us) values in global coordinates."""
    assert len(poses) == 7
    assert all(all(math.isfinite(v) for v in p) for p in poses)
    assert all(b[3] > a[3] for a, b in zip(poses, poses[1:])), "Non-increasing timestamps"
    x0, y0, yaw0, t0 = poses[0]
    lateral = [-(x - x0) * math.sin(yaw0) + (y - y0) * math.cos(yaw0) for x, y, _, _ in poses[1:]]
    lengths = [math.hypot(b[0] - a[0], b[1] - a[1]) for a, b in zip(poses, poses[1:])]
    yaw_steps = [abs(wrap(b[2] - a[2])) for a, b in zip(poses, poses[1:])]
    path_length = sum(lengths)
    state = "stopped" if path_length <= 0.3 else "low_speed" if path_length <= 1.5 else "moving"
    eligible = all(v > 0.3 for v in lengths)
    disp_cum = yaw_mid_cum = gap = None
    if eligible:
        directions = [math.atan2(b[1] - a[1], b[0] - a[0]) for a, b in zip(poses, poses[1:])]
        # Compare both methods on the same six segment midpoints (five differences).
        yaw_mid = [wrap(a[2] + wrap(b[2] - a[2]) / 2) for a, b in zip(poses, poses[1:])]
        disp_cum = math.degrees(sum(abs(wrap(b - a)) for a, b in zip(directions, directions[1:])))
        yaw_mid_cum = math.degrees(sum(abs(wrap(b - a)) for a, b in zip(yaw_mid, yaw_mid[1:])))
        gap = abs(disp_cum - yaw_mid_cum)
    return {
        "endpoint_lateral_m": lateral[-1],
        "abs_endpoint_lateral_m": abs(lateral[-1]),
        "max_abs_lateral_m": max(abs(v) for v in lateral),
        "path_length_m": path_length,
        "motion_state": state,
        "cum_yaw_deg": math.degrees(sum(yaw_steps)),
        "max_step_yaw_deg": math.degrees(max(yaw_steps)),
        "horizon_seconds": (poses[-1][3] - t0) / 1e6,
        "min_step_seconds": min((b[3] - a[3]) / 1e6 for a, b in zip(poses, poses[1:])),
        "max_step_seconds": max((b[3] - a[3]) / 1e6 for a, b in zip(poses, poses[1:])),
        "crosscheck_eligible": eligible,
        "displacement_cum_deg": disp_cum,
        "yaw_midpoint_cum_deg": yaw_mid_cum,
        "crosscheck_gap_deg": gap,
        "primary_low_lateral": abs(lateral[-1]) < 2.0,
        "endpoint_only_excursion": abs(lateral[-1]) < 2.0 and max(abs(v) for v in lateral) >= 2.0,
    }


def verify_official(hashes, log):
    """Stream official metadata without storing the large archive or sensor files."""
    result = {"url": OFFICIAL_META_URL, "status": "not_completed", "files": {}}
    try:
        req = urllib.request.Request(OFFICIAL_META_URL, headers={"User-Agent": "KnowledgeAnchor-motion-census"})
        with urllib.request.urlopen(req, timeout=90) as response:
            result["etag"] = response.headers.get("ETag")
            result["last_modified"] = response.headers.get("Last-Modified")
            result["content_length"] = response.headers.get("Content-Length")
            with tarfile.open(fileobj=response, mode="r|gz") as archive:
                for member in archive:
                    name = member.name.replace("\\", "/").split("/")[-1]
                    if member.isfile() and name in hashes:
                        assert name not in result["files"], "Duplicate official metadata member"
                        h = hashlib.sha256()
                        f = archive.extractfile(member)
                        for chunk in iter(lambda: f.read(1 << 20), b""):
                            h.update(chunk)
                        result["files"][name] = {"sha256": h.hexdigest(), "bytes": member.size,
                                                 "matches_mirror": h.hexdigest() == hashes[name]}
                        log("official metadata hashed: " + name)
        assert set(result["files"]) == set(hashes), "Official tables incomplete"
        result["status"] = "matched" if all(v["matches_mirror"] for v in result["files"].values()) else "mismatch"
    except Exception as exc:
        result["error"] = type(exc).__name__ + ": " + str(exc)
        log("official comparison incomplete: " + result["error"])
    return result


def load_records(meta, split_manifest, log):
    def load(name):
        with open(meta / name, encoding="utf-8") as f:
            return json.load(f)
    scenes, samples = load("scene.json"), load("sample.json")
    assert len(scenes) == 850 and len(samples) == 34149
    by_token = {v["token"]: v for v in samples}
    assert len(by_token) == len(samples)
    split_of = {name: split for split, names in split_manifest["splits"].items() for name in names}
    assert len(split_of) == 850 and {s["name"] for s in scenes} == set(split_of)
    sd = load("sample_data.json")
    sd_count = len(sd)
    lidar = {}
    for row in sd:
        parts = row.get("filename", "").split("__")
        if row["is_key_frame"] and len(parts) >= 3 and parts[1] == "LIDAR_TOP":
            tok = row["sample_token"]
            assert tok not in lidar, "Duplicate LIDAR keyframe"
            lidar[tok] = (row["ego_pose_token"], row["timestamp"])
    assert set(lidar) == set(by_token), "Missing or extra LIDAR keyframe"
    del sd
    gc.collect()
    needed = {v[0] for v in lidar.values()}
    ep = load("ego_pose.json")
    ep_count = len(ep)
    poses = {v["token"]: (v["translation"][0], v["translation"][1], quat_yaw(v["rotation"]), v["timestamp"])
             for v in ep if v["token"] in needed}
    assert set(poses) == needed
    del ep
    gc.collect()
    visited = set()
    records = []
    population = Counter()
    tails = Counter()
    for scene in sorted(scenes, key=lambda s: s["name"]):
        chain = []
        token = scene["first_sample_token"]
        while token:
            assert token not in visited, "Repeated sample or scene cycle"
            visited.add(token)
            row = by_token[token]
            assert row["scene_token"] == scene["token"]
            assert row["prev"] == (chain[-1] if chain else "")
            chain.append(token)
            token = row["next"]
        assert len(chain) == scene["nbr_samples"] and chain[-1] == scene["last_sample_token"]
        assert len(chain) >= 7
        split = split_of[scene["name"]]
        population[split] += len(chain)
        tails[split] += min(6, len(chain))
        for i in range(len(chain) - 6):
            tokens = chain[i:i + 7]
            window = []
            for tok in tokens:
                ep_token, timestamp = lidar[tok]
                pose = poses[ep_token]
                assert pose[3] == timestamp, "ego_pose/sample_data timestamp mismatch"
                window.append(pose)
            metrics = motion_metrics(window)
            records.append({"sample_token": tokens[0], "scene": scene["name"], "split": split, **metrics})
    assert visited == set(by_token)
    assert population == {"train": 28130, "val": 6019}
    assert Counter(v["split"] for v in records) == {"train": 23930, "val": 5119}
    log(f"complete windows={len(records)}; excluded tails={dict(tails)}")
    return records, {"scenes": len(scenes), "samples": len(samples), "sample_data": sd_count,
                     "ego_pose": ep_count, "all_samples_by_split": dict(population),
                     "excluded_incomplete_by_split": dict(tails), "missing_lidar": 0}


def summarize(rows):
    n = len(rows)
    counts = {
        "endpoint_lt_2m": sum(r["primary_low_lateral"] for r in rows),
        "max_lateral_lt_2m": sum(r["max_abs_lateral_m"] < 2 for r in rows),
        "endpoint_only_excursion": sum(r["endpoint_only_excursion"] for r in rows),
    }
    return {"n": n, "counts": counts,
            "ratios": {k: v / n if n else None for k, v in counts.items()},
            "majority_gt_half": counts["endpoint_lt_2m"] * 2 > n if n else None,
            "motion_state_counts": dict(Counter(r["motion_state"] for r in rows))}


def quantiles(values):
    values = sorted(values)
    if not values:
        return None
    return {"min": values[0], "median": statistics.median(values),
            "p05": values[round((len(values) - 1) * .05)],
            "p95": values[round((len(values) - 1) * .95)], "max": values[-1]}


def write_csv(path, rows):
    assert rows
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def run(meta_dir, out_dir, provenance, split_manifest):
    started = time.time()
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    logfile = out / "run-log.txt"
    logfile.write_text("", encoding="utf-8")
    def log(message):
        line = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()) + " " + message
        print(line, flush=True)
        with logfile.open("a", encoding="utf-8") as f:
            f.write(line + "\n")
    try:
        meta = Path(meta_dir)
        assert meta.name == "v1.0-trainval"
        log("started " + CLAIM_ID)
        hashes = {name: sha256_file(meta / name) for name in TABLES}
        official = verify_official(hashes, log)
        json_write(out / "official-comparison.json", official)
        assert official["status"] != "mismatch", "Mirror differs from official metadata; stop interpretation"
        records, counts = load_records(meta, split_manifest, log)
        write_csv(out / "sample-metrics.csv", records)
        populations = {}
        sensitivities = []
        for split in ("trainval", "train", "val"):
            base = [r for r in records if split == "trainval" or r["split"] == split]
            for state in ("all", "stopped", "low_speed", "moving", "non_stopped"):
                rows = [r for r in base if state == "all" or r["motion_state"] == state or
                        (state == "non_stopped" and r["motion_state"] != "stopped")]
                populations[f"{split}_{state}"] = summarize(rows)
                for threshold in PROTOCOL["lateral_sensitivity_m"]:
                    for field in ("abs_endpoint_lateral_m", "max_abs_lateral_m"):
                        hits = sum(r[field] < threshold for r in rows)
                        sensitivities.append({"split": split, "motion_state": state, "metric": field,
                                              "threshold": threshold, "count": hits, "n": len(rows),
                                              "ratio": hits / len(rows) if rows else None})
                for threshold in PROTOCOL["heading_sensitivity_deg"]:
                    hits = sum(r["cum_yaw_deg"] < threshold for r in rows)
                    sensitivities.append({"split": split, "motion_state": state, "metric": "cum_yaw_deg",
                                          "threshold": threshold, "count": hits, "n": len(rows),
                                          "ratio": hits / len(rows) if rows else None})
        write_csv(out / "sensitivity.csv", sensitivities)
        grouped = defaultdict(list)
        for row in records:
            grouped[row["scene"]].append(row)
        scene_rows = [{"scene": name, "split": rows[0]["split"], "n": len(rows),
                       "endpoint_lt_2m_count": sum(r["primary_low_lateral"] for r in rows),
                       "endpoint_lt_2m_ratio": sum(r["primary_low_lateral"] for r in rows) / len(rows),
                       "moving_n": sum(r["motion_state"] == "moving" for r in rows)}
                      for name, rows in sorted(grouped.items())]
        write_csv(out / "scene-metrics.csv", scene_rows)
        eligible = [r for r in records if r["crosscheck_eligible"]]
        conflicts = [r for r in eligible if r["crosscheck_gap_deg"] > 15]
        crosscheck = {"eligible_n": len(eligible), "excluded_short_step_n": len(records) - len(eligible),
                      "gap_gt_15deg_n": len(conflicts),
                      "gap_gt_15deg_ratio_of_eligible": len(conflicts) / len(eligible) if eligible else None,
                      "gap_deg_distribution": quantiles([r["crosscheck_gap_deg"] for r in eligible]),
                      "interpretation": "diagnostic only; compares accumulated directions at segment midpoints, not per-step orientation conflicts"}
        freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout
        (out / "pip-freeze.txt").write_text(freeze, encoding="utf-8")
        cpu = next((l.split(":", 1)[1].strip() for l in Path("/proc/cpuinfo").read_text().splitlines()
                    if l.startswith("model name")), "unknown") if Path("/proc/cpuinfo").exists() else platform.processor()
        result = {"claim_id": CLAIM_ID, "protocol": PROTOCOL, "provenance": provenance,
                  "environment": {"python": sys.version, "platform": platform.platform(), "cpu": cpu,
                                  "gpu_used": False, "pip_freeze_sha256": sha256_file(out / "pip-freeze.txt")},
                  "data": {"dataset": "sahangunasekara92/nuscenes-v1-0-full-keyframes", "version": "v1.0-trainval",
                           "sha256": hashes, "counts": counts, "official_comparison_status": official["status"]},
                  "populations": populations, "scene_level": {"n": len(scene_rows),
                       "equal_scene_mean_ratio": statistics.mean(r["endpoint_lt_2m_ratio"] for r in scene_rows),
                       "ratio_distribution": quantiles([r["endpoint_lt_2m_ratio"] for r in scene_rows])},
                  "crosscheck": crosscheck,
                  "timing": {"horizon_seconds": quantiles([r["horizon_seconds"] for r in records]),
                             "horizon_outside_2p75_3p25_n": sum(abs(r["horizon_seconds"] - 3) > .25 for r in records),
                             "min_step_seconds": min(r["min_step_seconds"] for r in records),
                             "max_step_seconds": max(r["max_step_seconds"] for r in records)},
                  "runtime_seconds": time.time() - started,
                  "limits": ["trainval only, excludes 150 test scenes", "endpoint low lateral displacement is not scene simplicity",
                             "does not test ego-status shortcut causality", "descriptive census, no independent held-out confirmation",
                             "strict thresholds; stopped and low_speed are operational path-length bins"]}
        json_write(out / "results.json", result)
        log("finished: " + json.dumps(populations["trainval_all"]))
        manifest = [f"{sha256_file(p)}  {p.name}" for p in sorted(out.iterdir()) if p.is_file() and p.name != "sha256-manifest.txt"]
        (out / "sha256-manifest.txt").write_text("\n".join(manifest) + "\n", encoding="utf-8")
        return result
    except Exception as exc:
        log("FAILED " + type(exc).__name__ + ": " + str(exc))
        raise


def locate_metadata(root="/kaggle/input"):
    found = []
    for directory, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d.lower() not in ("samples", "sweeps", "maps") and not d.startswith(".")]
        if Path(directory).name == "v1.0-trainval" and set(TABLES) <= set(files):
            found.append(directory)
            dirs[:] = []
    assert len(found) == 1, f"Expected one metadata source, got {found}"
    return found[0]


In [ ]:
"""Recompute ego motion and describe lateral displacement versus speed."""
import bisect
import csv
import gc
import gzip
import hashlib
import io
import json
import math
import os
import platform
import statistics
import subprocess
import sys
import tarfile
import tempfile
import time
import urllib.request
from collections import Counter
from pathlib import Path

SCOPES = ("all", "trainval", "train", "val", "test")
SPLITS = ("train", "val", "test")
METRICS = ("abs_endpoint_lateral_m", "max_abs_lateral_m", "mean_speed_mps", "start_speed_mps")
SPEEDS = ("start_speed_mps", "mean_speed_mps")
LATERAL_THRESHOLDS = (0.1, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0)
SPEED_EDGES = (0.0, 0.5, 2.0, 5.0, 10.0, 15.0, 20.0)
TAIL_CUTS = (5.0, 10.0, 15.0, 20.0)
QUANTILES = {"min": 0.0, "p10": 0.1, "p25": 0.25, "p50": 0.5,
             "p75": 0.75, "p90": 0.9, "p95": 0.95, "p99": 0.99, "max": 1.0}
TABLES = ("scene.json", "sample.json", "sample_data.json", "ego_pose.json")
TEST_URL = "https://motional-nuscenes.s3.amazonaws.com/public/v1.0/v1.0-test_meta.tgz"


def file_hash(path):
    result = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            result.update(block)
    return result.hexdigest()


def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")


def write_csv(path, rows, compressed=False):
    def write(stream):
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    if compressed:
        with open(path, "wb") as raw:
            with gzip.GzipFile(filename="", fileobj=raw, mode="wb", mtime=0) as gz:
                with io.TextIOWrapper(gz, encoding="utf-8", newline="") as stream:
                    write(stream)
    else:
        with open(path, "w", encoding="utf-8", newline="") as stream:
            write(stream)


def describe(values):
    values = sorted(v for v in values if v is not None)
    if not values:
        return {"n": 0, "mean": None, "population_std": None, **{key: None for key in QUANTILES}}
    assert all(math.isfinite(v) for v in values)
    result = {"n": len(values), "mean": statistics.mean(values), "population_std": statistics.pstdev(values)}
    for key, q in QUANTILES.items():
        h = (len(values) - 1) * q
        left = math.floor(h)
        right = math.ceil(h)
        result[key] = values[left] + (values[right] - values[left]) * (h - left)
    return result


def average_speed(metrics):
    assert metrics["horizon_seconds"] > 0
    return metrics["path_length_m"] / metrics["horizon_seconds"]


def past_speed(previous, current):
    if previous is None:
        return None, None
    dt = (current[3] - previous[3]) / 1e6
    assert dt > 0, "Previous keyframe timestamp is not earlier"
    return math.hypot(current[0] - previous[0], current[1] - previous[1]) / dt, dt


def subset(rows, scope):
    return [r for r in rows if scope == "all" or r["split"] == scope or
            scope == "trainval" and r["split"] in ("train", "val")]


def decode_baseline(path):
    with gzip.open(path, "rt", encoding="utf-8", newline="") as stream:
        rows = list(csv.DictReader(stream))
    for row in rows:
        for key, value in list(row.items()):
            if key in ("sample_token", "scene", "split", "motion_state"):
                continue
            if key in ("crosscheck_eligible", "primary_low_lateral", "endpoint_only_excursion"):
                assert value in ("True", "False")
                row[key] = value == "True"
            else:
                row[key] = float(value) if value else None
    return {r["sample_token"]: r for r in rows}


def find_inputs(root):
    metadata, baseline = [], []
    for directory, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d.lower() not in ("samples", "sweeps", "maps") and not d.startswith(".")]
        if Path(directory).name == "v1.0-trainval" and set(TABLES) <= set(files):
            metadata.append(Path(directory))
        if {"sample-metrics.csv.gz", "results.json"} <= set(files):
            baseline.append(Path(directory))
    assert len(metadata) == len(baseline) == 1, (metadata, baseline)
    return metadata[0], baseline[0]


def official_test(meta, expected_hashes, log):
    info = {"url": TEST_URL, "files": {}}
    request = urllib.request.Request(TEST_URL, headers={"User-Agent": "KnowledgeAnchor-motion-profile"})
    with urllib.request.urlopen(request, timeout=90) as response:
        info.update({"http_status": response.status, "etag": response.headers.get("ETag"),
                     "last_modified": response.headers.get("Last-Modified"), "content_length": response.headers.get("Content-Length")})
        with tarfile.open(fileobj=response, mode="r|gz") as archive:
            for member in archive:
                name = member.name.replace("\\", "/").split("/")[-1]
                if member.isfile() and name in TABLES:
                    assert name not in info["files"]
                    with (meta / name).open("wb") as target:
                        source = archive.extractfile(member)
                        for block in iter(lambda: source.read(1 << 20), b""):
                            target.write(block)
                    sha = file_hash(meta / name)
                    assert sha == expected_hashes[name], "Changed official test metadata: " + name
                    info["files"][name] = {"sha256": sha, "bytes": member.size}
                    log("test input matched previous official bytes: " + name)
    assert set(info["files"]) == set(TABLES)
    return info


def collect(meta, version, split_manifest, baseline, core, log):
    def load(name):
        with (meta / name).open(encoding="utf-8") as stream:
            return json.load(stream)
    scenes, samples = load("scene.json"), load("sample.json")
    names = {s["name"] for s in scenes}
    expected_names = set(split_manifest["splits"]["test"]) if version == "test" else set(split_manifest["splits"]["train"] + split_manifest["splits"]["val"])
    assert names == expected_names
    assert len(samples) == (6008 if version == "test" else 34149)
    by_token = {s["token"]: s for s in samples}
    assert len(by_token) == len(samples)
    sd = load("sample_data.json")
    lidar = {}
    for row in sd:
        channel = row.get("filename", "").split("__")
        if row["is_key_frame"] and len(channel) >= 3 and channel[1] == "LIDAR_TOP":
            assert row["sample_token"] not in lidar
            lidar[row["sample_token"]] = (row["ego_pose_token"], row["timestamp"])
    assert set(lidar) == set(by_token)
    del sd
    gc.collect()
    needed = {v[0] for v in lidar.values()}
    ep = load("ego_pose.json")
    poses = {e["token"]: (e["translation"][0], e["translation"][1], core.quat_yaw(e["rotation"]), e["timestamp"])
             for e in ep if e["token"] in needed}
    assert set(poses) == needed
    del ep
    gc.collect()
    split_of = {name: split for split, entries in split_manifest["splits"].items() for name in entries}
    records, visited, no_previous = [], set(), 0
    for scene in sorted(scenes, key=lambda s: s["name"]):
        chain, token = [], scene["first_sample_token"]
        while token:
            assert token not in visited
            visited.add(token)
            sample = by_token[token]
            assert sample["scene_token"] == scene["token"]
            assert sample["prev"] == (chain[-1] if chain else "")
            chain.append(token)
            token = sample["next"]
        assert len(chain) == scene["nbr_samples"] and len(chain) >= 7 and chain[-1] == scene["last_sample_token"]
        sequence = []
        for token in chain:
            ep_token, timestamp = lidar[token]
            pose = poses[ep_token]
            assert pose[3] == timestamp
            sequence.append(pose)
        for i in range(len(chain) - 6):
            original = {"sample_token": chain[i], "scene": scene["name"], "split": split_of[scene["name"]],
                        **core.motion_metrics(sequence[i:i + 7])}
            assert original == baseline[chain[i]], "Original per-sample measurement changed: " + chain[i]
            speed, dt = past_speed(sequence[i - 1] if i else None, sequence[i])
            no_previous += speed is None
            records.append({**original, "mean_speed_mps": average_speed(original), "start_speed_mps": speed,
                            "start_speed_interval_seconds": dt, "start_speed_missing_reason": "first_keyframe" if speed is None else ""})
    assert visited == set(by_token)
    assert len(records) == (5108 if version == "test" else 29049)
    assert no_previous == len(scenes)
    log(f"{version}: recomputed {len(records)} original rows exactly; start-speed missing={no_previous}")
    return records


def group_result(rows, scope, speed_metric, kind, lo, hi):
    eligible = [r for r in rows if r[speed_metric] is not None]
    group = [r for r in eligible if r[speed_metric] >= lo and (hi is None or r[speed_metric] < hi)]
    endpoint = describe([r["abs_endpoint_lateral_m"] for r in group])
    maximum = describe([r["max_abs_lateral_m"] for r in group])
    result = {"scope": scope, "speed_metric": speed_metric, "group_kind": kind, "speed_lo_mps": lo,
              "speed_hi_mps": hi, "n": len(group), "eligible_n": len(eligible), "missing_speed_n": len(rows) - len(eligible),
              "n_scenes": len({r["scene"] for r in group}), "fraction_of_speed_eligible": len(group) / len(eligible) if eligible else None,
              "endpoint_p50_m": endpoint["p50"], "endpoint_p90_m": endpoint["p90"], "endpoint_p95_m": endpoint["p95"],
              "maximum_p50_m": maximum["p50"], "maximum_p90_m": maximum["p90"], "maximum_p95_m": maximum["p95"]}
    for threshold in LATERAL_THRESHOLDS:
        for field, prefix in (("abs_endpoint_lateral_m", "endpoint"), ("max_abs_lateral_m", "maximum")):
            count = sum(r[field] < threshold for r in group)
            key = f"{prefix}_lt_{threshold:g}m"
            result[key + "_count"] = count
            result[key + "_ratio"] = count / len(group) if group else None
    return result


def full_ecdf(values):
    counts = Counter(v for v in values if v is not None)
    n = sum(counts.values())
    cumulative = 0
    rows = []
    for value, count in sorted(counts.items()):
        rows.append({"value": value, "count_lt": cumulative, "count_le": cumulative + count,
                     "n": n, "fraction_lt": cumulative / n, "fraction_le": (cumulative + count) / n})
        cumulative += count
    return rows


def run_profile(core, provenance, split_manifest, expected, input_root="/kaggle/input", output="/kaggle/working"):
    started = time.time()
    out = Path(output)
    out.mkdir(parents=True, exist_ok=True)
    logfile = out / "run-log.txt"
    logfile.write_text("", encoding="utf-8")
    def log(message):
        line = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()) + " " + message
        print(line, flush=True)
        with logfile.open("a", encoding="utf-8") as stream:
            stream.write(line + "\n")
    try:
        log("started full motion profile; no preselected conclusion")
        meta, source = find_inputs(input_root)
        for name, sha in expected["baseline_outputs"].items():
            assert file_hash(source / name) == sha, "Changed baseline output: " + name
        baseline = decode_baseline(source / "sample-metrics.csv.gz")
        assert len(baseline) == 34157
        trainval_hashes = {name: file_hash(meta / name) for name in TABLES}
        assert trainval_hashes == expected["trainval_json"]
        log("trainval metadata matches previous official hashes")
        rows = collect(meta, "trainval", split_manifest, baseline, core, log)
        with tempfile.TemporaryDirectory(prefix="nuscenes-profile-test-") as scratch:
            source_info = official_test(Path(scratch), expected["test_json"], log)
            rows += collect(Path(scratch), "test", split_manifest, baseline, core, log)
        assert len(rows) == len({r["sample_token"] for r in rows}) == 34157
        assert {r["sample_token"] for r in rows} == set(baseline)
        assert len({r["scene"] for r in rows}) == 1000
        assert sum(r["start_speed_mps"] is None for r in rows) == 1000
        source_info["trainval_sha256"] = trainval_hashes
        save_json(out / "input-integrity.json", source_info)
        summaries, thresholds, groups = [], [], []
        for scope in SCOPES:
            selected = subset(rows, scope)
            for metric in METRICS:
                summary = {"scope": scope, "metric": metric, "scope_n": len(selected), **describe([r[metric] for r in selected])}
                summary["missing_n"] = len(selected) - summary["n"]
                summaries.append(summary)
            for metric in ("abs_endpoint_lateral_m", "max_abs_lateral_m"):
                values = sorted(r[metric] for r in selected)
                for threshold in LATERAL_THRESHOLDS:
                    count = bisect.bisect_left(values, threshold)
                    thresholds.append({"scope": scope, "metric": metric, "threshold_m": threshold,
                                       "count_lt": count, "n": len(values), "fraction_lt": count / len(values)})
            for speed in SPEEDS:
                for i, lo in enumerate(SPEED_EDGES):
                    hi = SPEED_EDGES[i + 1] if i + 1 < len(SPEED_EDGES) else None
                    groups.append(group_result(selected, scope, speed, "bin", lo, hi))
                for lo in TAIL_CUTS:
                    groups.append(group_result(selected, scope, speed, "ge", lo, None))
        write_csv(out / "quantiles.csv", summaries)
        write_csv(out / "lateral-thresholds.csv", thresholds)
        write_csv(out / "speed-conditioned.csv", groups)
        for split in SPLITS:
            write_csv(out / f"profile-samples-{split}.csv.gz", subset(rows, split), compressed=True)
        for metric in METRICS:
            write_csv(out / f"ecdf-{metric}.csv.gz", full_ecdf([r[metric] for r in rows]), compressed=True)
        freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout
        (out / "pip-freeze.txt").write_text(freeze, encoding="utf-8")
        cpu = next((line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
                    if line.startswith("model name")), "unknown") if Path("/proc/cpuinfo").exists() else platform.processor()
        profile = {"analysis": "descriptive nuScenes ego-motion profile", "provenance": provenance,
                   "definitions": {"average_speed": "sum of six XY segment lengths / actual future duration",
                                   "start_speed": "previous-keyframe XY distance / actual preceding duration; first frame missing",
                                   "speed_edges_mps": SPEED_EDGES, "tail_cuts_mps": TAIL_CUTS,
                                   "lateral_thresholds_m": LATERAL_THRESHOLDS, "quantile_method": "linear h=(n-1)q",
                                   "cdf_boundary": "threshold tables use strict <; full ECDF stores both < and <="},
                   "coverage": {"scenes": 1000, "future_windows": len(rows), "start_speed_available": 33157,
                                "start_speed_missing_first_keyframe": 1000, "all_original_rows_equal_baseline": True},
                   "quantiles": summaries, "lateral_thresholds": thresholds, "speed_conditioned": groups,
                   "environment": {"python": sys.version, "platform": platform.platform(), "cpu": cpu, "gpu_used": False,
                                   "pip_freeze_sha256": file_hash(out / "pip-freeze.txt")},
                   "runtime_seconds": time.time() - started}
        save_json(out / "profile.json", profile)
        log("finished: " + json.dumps(profile["coverage"]))
        for summary in summaries:
            if summary["scope"] == "all":
                log("distribution " + json.dumps(summary))
        names = [p.name for p in out.iterdir() if p.name.startswith(("ecdf-", "profile-samples-"))]
        names += ["profile.json", "input-integrity.json", "quantiles.csv", "lateral-thresholds.csv", "speed-conditioned.csv", "pip-freeze.txt", "run-log.txt"]
        assert all((out / name).stat().st_size < 5_000_000 for name in names)
        (out / "sha256-manifest.txt").write_text("\n".join(f"{file_hash(out / name)}  {name}" for name in sorted(names)) + "\n", encoding="utf-8")
        return profile
    except Exception as exc:
        log("FAILED " + type(exc).__name__ + ": " + str(exc))
        raise


In [ ]:
import types
CORE = types.SimpleNamespace(motion_metrics=motion_metrics, quat_yaw=quat_yaw)
PROVENANCE = {'startup_git_head': 'd50d40679a4a904a3faf92e0b2e76208c0b94502', 'measurement_code_sha256': 'ce4dda3c13b64fe511f109366796edbf6795385f747a979deb08ee48089e27d7', 'analysis_code_sha256': '6b44896400b58eef0a90b367066b7c107b3852fe8c25c584c7e52eb438c41894', 'protocol_readme_sha256': 'ff4ba00fc7003286aa3d2f3d4a0552773eb45f3a6d9ecbebbb460769b916d867', 'splits_manifest_sha256': 'bb7373fc9d4a759b00b8e343a74c253f769bb8e6706047154486cf67f6fa766c', 'kaggle_kernel': 'cauliyulong/claim-nuscenes-motion-profile', 'expected_inputs': {'baseline_outputs': {'sample-metrics.csv.gz': 'd36aa8c40ee6c012746381faa0bd5de858009b7f312e37f541c478f3ccdb9f13', 'results.json': 'cf4f15e3f08854f8abff28fa6fd36588faeadd7fb5434abc0f64661e57d2c523'}, 'trainval_json': {'scene.json': 'bdc3211bfbfda652d5e9dab73c45c7132925ca84f869bf45e8e14edeafb1d20b', 'sample.json': '6035ac58b6e971622be2bb1be15b917e7cb4e05ae984d6b339c27c1699c4ad9d', 'sample_data.json': '6dcad49f0b9bd7b1cef04e2a0ff2ac2879b46b938e8d48153f00693cebb4de21', 'ego_pose.json': 'be12bd501f694b344628ba0680a37d6e1ec83e62b37f310c205c1dbe41cd09ff'}, 'test_json': {'ego_pose.json': 'cd58a4deacddb06a0f0f3835cda2414834c7c97445e33f78a3a6bd52f6780714', 'scene.json': '92bb751d8fb3ff292a59892754d530ba8b89bb9ddd009fff5ae02cf2c090e034', 'sample_data.json': 'ea3983384a0bae7c047602c188d8183db99c89ffd642bc045dccff760cd997cc', 'sample.json': 'c20c37aaeb4ea5a603b338c815c3466c2e955c45a73d72ab60d1df969dfd3200'}}}
SPLITS_MANIFEST = {'source_url': 'https://raw.githubusercontent.com/nutonomy/nuscenes-devkit/b40adc467b919192899405d9b77871afee8efa07/python-sdk/nuscenes/utils/splits.py', 'devkit_commit': 'b40adc467b919192899405d9b77871afee8efa07', 'source_sha256': 'eab6fa5e2536a2a85bd9451fb35771833e262b4b96319a6b26fee1dce8f4e2cd', 'splits': {'train': ['scene-0001', 'scene-0002', 'scene-0004', 'scene-0005', 'scene-0006', 'scene-0007', 'scene-0008', 'scene-0009', 'scene-0010', 'scene-0011', 'scene-0019', 'scene-0020', 'scene-0021', 'scene-0022', 'scene-0023', 'scene-0024', 'scene-0025', 'scene-0026', 'scene-0027', 'scene-0028', 'scene-0029', 'scene-0030', 'scene-0031', 'scene-0032', 'scene-0033', 'scene-0034', 'scene-0041', 'scene-0042', 'scene-0043', 'scene-0044', 'scene-0045', 'scene-0046', 'scene-0047', 'scene-0048', 'scene-0049', 'scene-0050', 'scene-0051', 'scene-0052', 'scene-0053', 'scene-0054', 'scene-0055', 'scene-0056', 'scene-0057', 'scene-0058', 'scene-0059', 'scene-0060', 'scene-0061', 'scene-0062', 'scene-0063', 'scene-0064', 'scene-0065', 'scene-0066', 'scene-0067', 'scene-0068', 'scene-0069', 'scene-0070', 'scene-0071', 'scene-0072', 'scene-0073', 'scene-0074', 'scene-0075', 'scene-0076', 'scene-0120', 'scene-0121', 'scene-0122', 'scene-0123', 'scene-0124', 'scene-0125', 'scene-0126', 'scene-0127', 'scene-0128', 'scene-0129', 'scene-0130', 'scene-0131', 'scene-0132', 'scene-0133', 'scene-0134', 'scene-0135', 'scene-0138', 'scene-0139', 'scene-0149', 'scene-0150', 'scene-0151', 'scene-0152', 'scene-0154', 'scene-0155', 'scene-0157', 'scene-0158', 'scene-0159', 'scene-0160', 'scene-0161', 'scene-0162', 'scene-0163', 'scene-0164', 'scene-0165', 'scene-0166', 'scene-0167', 'scene-0168', 'scene-0170', 'scene-0171', 'scene-0172', 'scene-0173', 'scene-0174', 'scene-0175', 'scene-0176', 'scene-0177', 'scene-0178', 'scene-0179', 'scene-0180', 'scene-0181', 'scene-0182', 'scene-0183', 'scene-0184', 'scene-0185', 'scene-0187', 'scene-0188', 'scene-0190', 'scene-0191', 'scene-0192', 'scene-0193', 'scene-0194', 'scene-0195', 'scene-0196', 'scene-0199', 'scene-0200', 'scene-0202', 'scene-0203', 'scene-0204', 'scene-0206', 'scene-0207', 'scene-0208', 'scene-0209', 'scene-0210', 'scene-0211', 'scene-0212', 'scene-0213', 'scene-0214', 'scene-0218', 'scene-0219', 'scene-0220', 'scene-0222', 'scene-0224', 'scene-0225', 'scene-0226', 'scene-0227', 'scene-0228', 'scene-0229', 'scene-0230', 'scene-0231', 'scene-0232', 'scene-0233', 'scene-0234', 'scene-0235', 'scene-0236', 'scene-0237', 'scene-0238', 'scene-0239', 'scene-0240', 'scene-0241', 'scene-0242', 'scene-0243', 'scene-0244', 'scene-0245', 'scene-0246', 'scene-0247', 'scene-0248', 'scene-0249', 'scene-0250', 'scene-0251', 'scene-0252', 'scene-0253', 'scene-0254', 'scene-0255', 'scene-0256', 'scene-0257', 'scene-0258', 'scene-0259', 'scene-0260', 'scene-0261', 'scene-0262', 'scene-0263', 'scene-0264', 'scene-0283', 'scene-0284', 'scene-0285', 'scene-0286', 'scene-0287', 'scene-0288', 'scene-0289', 'scene-0290', 'scene-0291', 'scene-0292', 'scene-0293', 'scene-0294', 'scene-0295', 'scene-0296', 'scene-0297', 'scene-0298', 'scene-0299', 'scene-0300', 'scene-0301', 'scene-0302', 'scene-0303', 'scene-0304', 'scene-0305', 'scene-0306', 'scene-0315', 'scene-0316', 'scene-0317', 'scene-0318', 'scene-0321', 'scene-0323', 'scene-0324', 'scene-0328', 'scene-0347', 'scene-0348', 'scene-0349', 'scene-0350', 'scene-0351', 'scene-0352', 'scene-0353', 'scene-0354', 'scene-0355', 'scene-0356', 'scene-0357', 'scene-0358', 'scene-0359', 'scene-0360', 'scene-0361', 'scene-0362', 'scene-0363', 'scene-0364', 'scene-0365', 'scene-0366', 'scene-0367', 'scene-0368', 'scene-0369', 'scene-0370', 'scene-0371', 'scene-0372', 'scene-0373', 'scene-0374', 'scene-0375', 'scene-0376', 'scene-0377', 'scene-0378', 'scene-0379', 'scene-0380', 'scene-0381', 'scene-0382', 'scene-0383', 'scene-0384', 'scene-0385', 'scene-0386', 'scene-0388', 'scene-0389', 'scene-0390', 'scene-0391', 'scene-0392', 'scene-0393', 'scene-0394', 'scene-0395', 'scene-0396', 'scene-0397', 'scene-0398', 'scene-0399', 'scene-0400', 'scene-0401', 'scene-0402', 'scene-0403', 'scene-0405', 'scene-0406', 'scene-0407', 'scene-0408', 'scene-0410', 'scene-0411', 'scene-0412', 'scene-0413', 'scene-0414', 'scene-0415', 'scene-0416', 'scene-0417', 'scene-0418', 'scene-0419', 'scene-0420', 'scene-0421', 'scene-0422', 'scene-0423', 'scene-0424', 'scene-0425', 'scene-0426', 'scene-0427', 'scene-0428', 'scene-0429', 'scene-0430', 'scene-0431', 'scene-0432', 'scene-0433', 'scene-0434', 'scene-0435', 'scene-0436', 'scene-0437', 'scene-0438', 'scene-0439', 'scene-0440', 'scene-0441', 'scene-0442', 'scene-0443', 'scene-0444', 'scene-0445', 'scene-0446', 'scene-0447', 'scene-0448', 'scene-0449', 'scene-0450', 'scene-0451', 'scene-0452', 'scene-0453', 'scene-0454', 'scene-0455', 'scene-0456', 'scene-0457', 'scene-0458', 'scene-0459', 'scene-0461', 'scene-0462', 'scene-0463', 'scene-0464', 'scene-0465', 'scene-0467', 'scene-0468', 'scene-0469', 'scene-0471', 'scene-0472', 'scene-0474', 'scene-0475', 'scene-0476', 'scene-0477', 'scene-0478', 'scene-0479', 'scene-0480', 'scene-0499', 'scene-0500', 'scene-0501', 'scene-0502', 'scene-0504', 'scene-0505', 'scene-0506', 'scene-0507', 'scene-0508', 'scene-0509', 'scene-0510', 'scene-0511', 'scene-0512', 'scene-0513', 'scene-0514', 'scene-0515', 'scene-0517', 'scene-0518', 'scene-0525', 'scene-0526', 'scene-0527', 'scene-0528', 'scene-0529', 'scene-0530', 'scene-0531', 'scene-0532', 'scene-0533', 'scene-0534', 'scene-0535', 'scene-0536', 'scene-0537', 'scene-0538', 'scene-0539', 'scene-0541', 'scene-0542', 'scene-0543', 'scene-0544', 'scene-0545', 'scene-0546', 'scene-0566', 'scene-0568', 'scene-0570', 'scene-0571', 'scene-0572', 'scene-0573', 'scene-0574', 'scene-0575', 'scene-0576', 'scene-0577', 'scene-0578', 'scene-0580', 'scene-0582', 'scene-0583', 'scene-0584', 'scene-0585', 'scene-0586', 'scene-0587', 'scene-0588', 'scene-0589', 'scene-0590', 'scene-0591', 'scene-0592', 'scene-0593', 'scene-0594', 'scene-0595', 'scene-0596', 'scene-0597', 'scene-0598', 'scene-0599', 'scene-0600', 'scene-0639', 'scene-0640', 'scene-0641', 'scene-0642', 'scene-0643', 'scene-0644', 'scene-0645', 'scene-0646', 'scene-0647', 'scene-0648', 'scene-0649', 'scene-0650', 'scene-0651', 'scene-0652', 'scene-0653', 'scene-0654', 'scene-0655', 'scene-0656', 'scene-0657', 'scene-0658', 'scene-0659', 'scene-0660', 'scene-0661', 'scene-0662', 'scene-0663', 'scene-0664', 'scene-0665', 'scene-0666', 'scene-0667', 'scene-0668', 'scene-0669', 'scene-0670', 'scene-0671', 'scene-0672', 'scene-0673', 'scene-0674', 'scene-0675', 'scene-0676', 'scene-0677', 'scene-0678', 'scene-0679', 'scene-0681', 'scene-0683', 'scene-0684', 'scene-0685', 'scene-0686', 'scene-0687', 'scene-0688', 'scene-0689', 'scene-0695', 'scene-0696', 'scene-0697', 'scene-0698', 'scene-0700', 'scene-0701', 'scene-0703', 'scene-0704', 'scene-0705', 'scene-0706', 'scene-0707', 'scene-0708', 'scene-0709', 'scene-0710', 'scene-0711', 'scene-0712', 'scene-0713', 'scene-0714', 'scene-0715', 'scene-0716', 'scene-0717', 'scene-0718', 'scene-0719', 'scene-0726', 'scene-0727', 'scene-0728', 'scene-0730', 'scene-0731', 'scene-0733', 'scene-0734', 'scene-0735', 'scene-0736', 'scene-0737', 'scene-0738', 'scene-0739', 'scene-0740', 'scene-0741', 'scene-0744', 'scene-0746', 'scene-0747', 'scene-0749', 'scene-0750', 'scene-0751', 'scene-0752', 'scene-0757', 'scene-0758', 'scene-0759', 'scene-0760', 'scene-0761', 'scene-0762', 'scene-0763', 'scene-0764', 'scene-0765', 'scene-0767', 'scene-0768', 'scene-0769', 'scene-0786', 'scene-0787', 'scene-0789', 'scene-0790', 'scene-0791', 'scene-0792', 'scene-0803', 'scene-0804', 'scene-0805', 'scene-0806', 'scene-0808', 'scene-0809', 'scene-0810', 'scene-0811', 'scene-0812', 'scene-0813', 'scene-0815', 'scene-0816', 'scene-0817', 'scene-0819', 'scene-0820', 'scene-0821', 'scene-0822', 'scene-0847', 'scene-0848', 'scene-0849', 'scene-0850', 'scene-0851', 'scene-0852', 'scene-0853', 'scene-0854', 'scene-0855', 'scene-0856', 'scene-0858', 'scene-0860', 'scene-0861', 'scene-0862', 'scene-0863', 'scene-0864', 'scene-0865', 'scene-0866', 'scene-0868', 'scene-0869', 'scene-0870', 'scene-0871', 'scene-0872', 'scene-0873', 'scene-0875', 'scene-0876', 'scene-0877', 'scene-0878', 'scene-0880', 'scene-0882', 'scene-0883', 'scene-0884', 'scene-0885', 'scene-0886', 'scene-0887', 'scene-0888', 'scene-0889', 'scene-0890', 'scene-0891', 'scene-0892', 'scene-0893', 'scene-0894', 'scene-0895', 'scene-0896', 'scene-0897', 'scene-0898', 'scene-0899', 'scene-0900', 'scene-0901', 'scene-0902', 'scene-0903', 'scene-0945', 'scene-0947', 'scene-0949', 'scene-0952', 'scene-0953', 'scene-0955', 'scene-0956', 'scene-0957', 'scene-0958', 'scene-0959', 'scene-0960', 'scene-0961', 'scene-0975', 'scene-0976', 'scene-0977', 'scene-0978', 'scene-0979', 'scene-0980', 'scene-0981', 'scene-0982', 'scene-0983', 'scene-0984', 'scene-0988', 'scene-0989', 'scene-0990', 'scene-0991', 'scene-0992', 'scene-0994', 'scene-0995', 'scene-0996', 'scene-0997', 'scene-0998', 'scene-0999', 'scene-1000', 'scene-1001', 'scene-1002', 'scene-1003', 'scene-1004', 'scene-1005', 'scene-1006', 'scene-1007', 'scene-1008', 'scene-1009', 'scene-1010', 'scene-1011', 'scene-1012', 'scene-1013', 'scene-1014', 'scene-1015', 'scene-1016', 'scene-1017', 'scene-1018', 'scene-1019', 'scene-1020', 'scene-1021', 'scene-1022', 'scene-1023', 'scene-1024', 'scene-1025', 'scene-1044', 'scene-1045', 'scene-1046', 'scene-1047', 'scene-1048', 'scene-1049', 'scene-1050', 'scene-1051', 'scene-1052', 'scene-1053', 'scene-1054', 'scene-1055', 'scene-1056', 'scene-1057', 'scene-1058', 'scene-1074', 'scene-1075', 'scene-1076', 'scene-1077', 'scene-1078', 'scene-1079', 'scene-1080', 'scene-1081', 'scene-1082', 'scene-1083', 'scene-1084', 'scene-1085', 'scene-1086', 'scene-1087', 'scene-1088', 'scene-1089', 'scene-1090', 'scene-1091', 'scene-1092', 'scene-1093', 'scene-1094', 'scene-1095', 'scene-1096', 'scene-1097', 'scene-1098', 'scene-1099', 'scene-1100', 'scene-1101', 'scene-1102', 'scene-1104', 'scene-1105', 'scene-1106', 'scene-1107', 'scene-1108', 'scene-1109', 'scene-1110'], 'val': ['scene-0003', 'scene-0012', 'scene-0013', 'scene-0014', 'scene-0015', 'scene-0016', 'scene-0017', 'scene-0018', 'scene-0035', 'scene-0036', 'scene-0038', 'scene-0039', 'scene-0092', 'scene-0093', 'scene-0094', 'scene-0095', 'scene-0096', 'scene-0097', 'scene-0098', 'scene-0099', 'scene-0100', 'scene-0101', 'scene-0102', 'scene-0103', 'scene-0104', 'scene-0105', 'scene-0106', 'scene-0107', 'scene-0108', 'scene-0109', 'scene-0110', 'scene-0221', 'scene-0268', 'scene-0269', 'scene-0270', 'scene-0271', 'scene-0272', 'scene-0273', 'scene-0274', 'scene-0275', 'scene-0276', 'scene-0277', 'scene-0278', 'scene-0329', 'scene-0330', 'scene-0331', 'scene-0332', 'scene-0344', 'scene-0345', 'scene-0346', 'scene-0519', 'scene-0520', 'scene-0521', 'scene-0522', 'scene-0523', 'scene-0524', 'scene-0552', 'scene-0553', 'scene-0554', 'scene-0555', 'scene-0556', 'scene-0557', 'scene-0558', 'scene-0559', 'scene-0560', 'scene-0561', 'scene-0562', 'scene-0563', 'scene-0564', 'scene-0565', 'scene-0625', 'scene-0626', 'scene-0627', 'scene-0629', 'scene-0630', 'scene-0632', 'scene-0633', 'scene-0634', 'scene-0635', 'scene-0636', 'scene-0637', 'scene-0638', 'scene-0770', 'scene-0771', 'scene-0775', 'scene-0777', 'scene-0778', 'scene-0780', 'scene-0781', 'scene-0782', 'scene-0783', 'scene-0784', 'scene-0794', 'scene-0795', 'scene-0796', 'scene-0797', 'scene-0798', 'scene-0799', 'scene-0800', 'scene-0802', 'scene-0904', 'scene-0905', 'scene-0906', 'scene-0907', 'scene-0908', 'scene-0909', 'scene-0910', 'scene-0911', 'scene-0912', 'scene-0913', 'scene-0914', 'scene-0915', 'scene-0916', 'scene-0917', 'scene-0919', 'scene-0920', 'scene-0921', 'scene-0922', 'scene-0923', 'scene-0924', 'scene-0925', 'scene-0926', 'scene-0927', 'scene-0928', 'scene-0929', 'scene-0930', 'scene-0931', 'scene-0962', 'scene-0963', 'scene-0966', 'scene-0967', 'scene-0968', 'scene-0969', 'scene-0971', 'scene-0972', 'scene-1059', 'scene-1060', 'scene-1061', 'scene-1062', 'scene-1063', 'scene-1064', 'scene-1065', 'scene-1066', 'scene-1067', 'scene-1068', 'scene-1069', 'scene-1070', 'scene-1071', 'scene-1072', 'scene-1073'], 'test': ['scene-0077', 'scene-0078', 'scene-0079', 'scene-0080', 'scene-0081', 'scene-0082', 'scene-0083', 'scene-0084', 'scene-0085', 'scene-0086', 'scene-0087', 'scene-0088', 'scene-0089', 'scene-0090', 'scene-0091', 'scene-0111', 'scene-0112', 'scene-0113', 'scene-0114', 'scene-0115', 'scene-0116', 'scene-0117', 'scene-0118', 'scene-0119', 'scene-0140', 'scene-0142', 'scene-0143', 'scene-0144', 'scene-0145', 'scene-0146', 'scene-0147', 'scene-0148', 'scene-0265', 'scene-0266', 'scene-0279', 'scene-0280', 'scene-0281', 'scene-0282', 'scene-0307', 'scene-0308', 'scene-0309', 'scene-0310', 'scene-0311', 'scene-0312', 'scene-0313', 'scene-0314', 'scene-0333', 'scene-0334', 'scene-0335', 'scene-0336', 'scene-0337', 'scene-0338', 'scene-0339', 'scene-0340', 'scene-0341', 'scene-0342', 'scene-0343', 'scene-0481', 'scene-0482', 'scene-0483', 'scene-0484', 'scene-0485', 'scene-0486', 'scene-0487', 'scene-0488', 'scene-0489', 'scene-0490', 'scene-0491', 'scene-0492', 'scene-0493', 'scene-0494', 'scene-0495', 'scene-0496', 'scene-0497', 'scene-0498', 'scene-0547', 'scene-0548', 'scene-0549', 'scene-0550', 'scene-0551', 'scene-0601', 'scene-0602', 'scene-0603', 'scene-0604', 'scene-0606', 'scene-0607', 'scene-0608', 'scene-0609', 'scene-0610', 'scene-0611', 'scene-0612', 'scene-0613', 'scene-0614', 'scene-0615', 'scene-0616', 'scene-0617', 'scene-0618', 'scene-0619', 'scene-0620', 'scene-0621', 'scene-0622', 'scene-0623', 'scene-0624', 'scene-0827', 'scene-0828', 'scene-0829', 'scene-0830', 'scene-0831', 'scene-0833', 'scene-0834', 'scene-0835', 'scene-0836', 'scene-0837', 'scene-0838', 'scene-0839', 'scene-0840', 'scene-0841', 'scene-0842', 'scene-0844', 'scene-0845', 'scene-0846', 'scene-0932', 'scene-0933', 'scene-0935', 'scene-0936', 'scene-0937', 'scene-0938', 'scene-0939', 'scene-0940', 'scene-0941', 'scene-0942', 'scene-0943', 'scene-1026', 'scene-1027', 'scene-1028', 'scene-1029', 'scene-1030', 'scene-1031', 'scene-1032', 'scene-1033', 'scene-1034', 'scene-1035', 'scene-1036', 'scene-1037', 'scene-1038', 'scene-1039', 'scene-1040', 'scene-1041', 'scene-1042', 'scene-1043']}}
EXPECTED = {'baseline_outputs': {'sample-metrics.csv.gz': 'd36aa8c40ee6c012746381faa0bd5de858009b7f312e37f541c478f3ccdb9f13', 'results.json': 'cf4f15e3f08854f8abff28fa6fd36588faeadd7fb5434abc0f64661e57d2c523'}, 'trainval_json': {'scene.json': 'bdc3211bfbfda652d5e9dab73c45c7132925ca84f869bf45e8e14edeafb1d20b', 'sample.json': '6035ac58b6e971622be2bb1be15b917e7cb4e05ae984d6b339c27c1699c4ad9d', 'sample_data.json': '6dcad49f0b9bd7b1cef04e2a0ff2ac2879b46b938e8d48153f00693cebb4de21', 'ego_pose.json': 'be12bd501f694b344628ba0680a37d6e1ec83e62b37f310c205c1dbe41cd09ff'}, 'test_json': {'ego_pose.json': 'cd58a4deacddb06a0f0f3835cda2414834c7c97445e33f78a3a6bd52f6780714', 'scene.json': '92bb751d8fb3ff292a59892754d530ba8b89bb9ddd009fff5ae02cf2c090e034', 'sample_data.json': 'ea3983384a0bae7c047602c188d8183db99c89ffd642bc045dccff760cd997cc', 'sample.json': 'c20c37aaeb4ea5a603b338c815c3466c2e955c45a73d72ab60d1df969dfd3200'}}
RESULT = run_profile(CORE, PROVENANCE, SPLITS_MANIFEST, EXPECTED)
